In [1]:
from pathlib import Path
import datetime

import torch
import torch.nn.functional as F
from jaxtyping import Int

# custom utils
from muutils.misc import shorten_numerical_to_str
from trnbl import TrainingManager

# from trnbl.loggers.local import LocalLogger
from trnbl.loggers.tensorboard import TensorBoardLogger


from attention_motifs.ae import AttnAEConfig, AttnAE, contrastive_loss
from attention_motifs.dataset.dataset import CollectedAttentionPatternDataloader
from attention_motifs.dataset.util import AttentionPatternMetadata
from attention_motifs.profiling import TrainingProfiler, ProfilerMode

f:\projects\attention-motifs\.venv\Lib\site-packages\trnbl\loggers\base.py:17: UserWarning: GPUtil not available: No module named 'GPUtil'
  warnings.warn(f"GPUtil not available: {e}")


In [2]:
# magic autoreload
%load_ext autoreload
%autoreload 2

In [3]:
BATCH_SIZE: int = 4
N_TRAIN_BATCHES: int = 10
N_VAL_BATCHES: int = 10

In [4]:
TRAIN_LOADER_DATASET = CollectedAttentionPatternDataloader.read(
	"../data/activations/pile_5"
)
VAL_LOADER_DATASET = CollectedAttentionPatternDataloader.read(
	"../data/activations/pile_5_val"
)

TRAIN_LOADER = TRAIN_LOADER_DATASET.dataloader(
	BATCH_SIZE, shuffle=True, max_batches=N_TRAIN_BATCHES
)
VAL_LOADER = VAL_LOADER_DATASET.dataloader(
	BATCH_SIZE, shuffle=True, max_batches=N_VAL_BATCHES
)

print(f"Train loader: {len(TRAIN_LOADER)} batches, {len(TRAIN_LOADER.dataset)} samples")

Train loader: 10 batches, 40 samples


In [5]:
MODEL: AttnAE = AttnAE(
	AttnAEConfig(
		latent_dim=128,
	)
)

MODEL_CONFIG: AttnAEConfig = MODEL.config

model_n_params: int = sum(p.numel() for p in MODEL.parameters())
print(
	f"model has {model_n_params} ({shorten_numerical_to_str(model_n_params)}) parameters"
)

# model

model has 446273 (446K) parameters


In [6]:
TRAIN_LOADER: torch.utils.data.DataLoader
VAL_LOADER: torch.utils.data.DataLoader | None = None

PROJECT_NAME: str = "contrastive-ae"
CHECKPT_INTERVAL: str = "1/2 run"
EVAL_INTERVAL: str = "1/2 run"
DEVICE: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL = MODEL.to(DEVICE)

In [7]:
OPTIMIZER: torch.optim.Optimizer
LR_SCHEDULER: torch.optim.lr_scheduler._LRScheduler
OPTIMIZER, LR_SCHEDULER = MODEL_CONFIG.get_optim_and_lrs(MODEL)

In [8]:
def evaluation_step(model: AttnAE) -> dict[str, float]:
	"""Evaluate model on validation set"""
	if VAL_LOADER is None:
		return {}

	model.eval()
	val_metrics: dict[str, float] = {
		"val/loss": 0.0,
		"val/recon_loss": 0.0,
		"val/contrast_loss": 0.0,
	}

	with torch.no_grad():
		for patterns, metadata in VAL_LOADER:
			patterns = patterns.to(DEVICE).to(torch.float32).unsqueeze(1)
			x_recon, embeddings = model(patterns)

			# reconstruction loss
			recon_loss = F.mse_loss(x_recon, patterns)

			# contrastive loss using all pairs in batch
			# compute "classes" for contrastive loss
			# classes is a tensor of the same shape as the batch, where each element is an integer
			classes: Int[torch.Tensor, " batch"] = (
				AttentionPatternMetadata.contrastive_classes(metadata).to(DEVICE)
			)

			# compute contrastive loss
			contrast_loss = contrastive_loss(
				embeddings, classes, temperature=model.config.contrast_temperature
			)

			# combined loss and backward pass
			total_loss = (
				model.config.recon_weight * recon_loss
				+ model.config.contrast_weight * contrast_loss
			)
			val_metrics["val/loss"] += total_loss.item()
			val_metrics["val/recon_loss"] += recon_loss.item()
			val_metrics["val/contrast_loss"] += contrast_loss.item()

	for k in val_metrics:
		val_metrics[k] /= len(VAL_LOADER)

	model.train()
	return val_metrics

In [9]:
# setup logger
LOGGER: TensorBoardLogger = TensorBoardLogger(
	log_dir=Path("tb-logs-convAE"),
	name=PROJECT_NAME + datetime.datetime.now().strftime("-%Y-%m-%d-%H-%M-%S"),
	# metric_names=[
	# 	"train/loss",
	# 	"train/recon_loss",
	# 	"train/contrast_loss",
	# 	"val/loss",
	# 	"val/recon_loss",
	# 	"val/contrast_loss",
	# ],
	train_config=dict(
		model_config=MODEL.zanj_model_config.serialize(),
		model_str=str(MODEL),
	),
)

profiler = TrainingProfiler(mode=ProfilerMode.FULL, output_dir=Path("profiles"))

with profiler.profile("training"):
	with TrainingManager(
		model=MODEL,
		logger=LOGGER,
		evals={
			EVAL_INTERVAL: evaluation_step,
		}.items(),
		checkpoint_interval=CHECKPT_INTERVAL,
	) as tr:
		for epoch in tr.epoch_loop(range(MODEL_CONFIG.num_epochs)):
			for patterns, metadata in tr.batch_loop(TRAIN_LOADER):
				this_batch_size: int = len(metadata)

				# move to device
				patterns = patterns.to(DEVICE).to(torch.float32).unsqueeze(1)

				assert patterns.shape[0] == this_batch_size

				# reset gradients
				OPTIMIZER.zero_grad()

				# forward pass
				x_recon, embeddings = MODEL(patterns)

				# reconstruction loss
				recon_loss = F.mse_loss(x_recon, patterns)

				# compute "classes" for contrastive loss
				# classes is a tensor of the same shape as the batch, where each element is an integer
				classes: Int[torch.Tensor, " batch"] = (
					AttentionPatternMetadata.contrastive_classes(metadata).to(DEVICE)
				)

				# compute contrastive loss
				contrast_loss = contrastive_loss(embeddings, classes)

				# combined loss and backward pass
				total_loss = (
					MODEL_CONFIG.recon_weight * recon_loss
					+ MODEL_CONFIG.contrast_weight * contrast_loss
				)

				# Memory snapshot after forward pass
				profiler.snapshot()

				# backward pass
				total_loss.backward()
				OPTIMIZER.step()
				LR_SCHEDULER.step(epoch)

				# log metrics
				metrics: dict[str, float] = {
					"train/loss": total_loss.item() / this_batch_size,
					"train/recon_loss": recon_loss.item() / this_batch_size,
					"train/contrast_loss": contrast_loss.item() / this_batch_size,
					"lr": LR_SCHEDULER.get_last_lr()[0],
				}
				tr.batch_update(
					samples=len(metadata),
					**metrics,
				)

				# cleanup
				del patterns, x_recon, embeddings, recon_loss, contrast_loss, total_loss

				# Memory snapshot after cleanup
				profiler.snapshot()


# print summary
profiler.print_summary()

starting training manager initialization


training run:   0%|          | 0/5 [00:00<?, ? epochs/s]

initialized training manager


F:\projects\attention-motifs\attention_motifs\ae.py:523: UserWarning: No positive pairs found in batch
  warnings.warn("No positive pairs found in batch")
training run: 100%|██████████| 5/5 [02:09<00:00, 25.96s/ epochs]


training complete

Profile: training
PROFILING RESULTS

Total time: 133.63 seconds

Memory Usage Summary:
Peak CPU: 9692.4 MB
Peak GPU: 4772.1 MB

CPU Memory (MB):
  0th percentile: 2465.3
  25th percentile: 8882.7
  50th percentile: 9139.8
  75th percentile: 9635.5
  100th percentile: 9692.4

GPU Memory (MB):
  0th percentile: 1.7
  25th percentile: 24.8
  50th percentile: 24.8
  75th percentile: 690.8
  100th percentile: 4772.1


F:\projects\attention-motifs\attention_motifs\profiling.py:243: UserWarning: Error during profiling: !stack.empty() INTERNAL ASSERT FAILED at "C:\\actions-runner\\_work\\pytorch\\pytorch\\pytorch\\torch\\csrc\\autograd\\profiler_python.cpp":983, please report a bug to PyTorch. Python replay stack is empty.
  warnings.warn(f"Error during profiling: {e}")
F:\projects\attention-motifs\attention_motifs\profiling.py:260: UserWarning: Error saving GPU trace: 'NoneType' object has no attribute 'save'
  warnings.warn(f"Error saving GPU trace: {e}")
